## Imports

In [99]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from implicit.als import AlternatingLeastSquares
import faiss

## Configuration

In [100]:
RANDOM_STATE = 42
TOP_K = 10
CANDIDATE_K = 100
FINAL_CANDIDATE_K = 300
SEMANTIC_RETRIEVAL_K = 100
TFIDF_RETRIEVAL_K = 100
ITEM_CF_RETRIEVAL_K = 100
ALS_RETRIEVAL_K = 100
POPULARITY_K = 100
TRENDING_K = 100
COUNTRY_K = 100
RECENT_DAYS = 7
RECENCY_HALF_LIFE_DAYS = 14
MIN_INTERACTIONS_FOR_PERSONALIZATION = 2

## Paths

In [101]:
PROJECT_DIR = Path(r"D:\iPrint-News-Recommendation-Ranking-System")
DATA_DIR = PROJECT_DIR / "Dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
FEATURE_DIR = ARTIFACT_DIR / "features"
INDEX_DIR = ARTIFACT_DIR / "indexes"
SEMANTIC_DIR = ARTIFACT_DIR / "semantic"
CANDIDATE_DIR = ARTIFACT_DIR / "candidates"
for directory in [ARTIFACT_DIR,FEATURE_DIR,INDEX_DIR,SEMANTIC_DIR,CANDIDATE_DIR,]:
    directory.mkdir(parents=True,exist_ok=True)
print("Project directory:", PROJECT_DIR)
print("Candidate directory:", CANDIDATE_DIR)

Project directory: D:\iPrint-News-Recommendation-Ranking-System
Candidate directory: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates


## Load Data

In [102]:
consumer = pd.read_csv(DATA_DIR / "consumer_transanctions.csv")
content = pd.read_csv(DATA_DIR / "platform_content.csv")
print("Consumer shape:", consumer.shape)
print("Content shape:", content.shape)

Consumer shape: (72312, 8)
Content shape: (3122, 13)


## Validation

In [103]:
required_consumer_columns = ["event_timestamp","interaction_type","item_id","consumer_id","consumer_session_id","consumer_device_info","consumer_location","country",]
required_content_columns = ["event_timestamp","interaction_type","item_id","producer_id","producer_session_id","producer_device_info","producer_location","producer_country","item_type","item_url","title","text_description","language",]
missing_consumer = [col for col in required_consumer_columns if col not in consumer.columns]
missing_content = [col for col in required_content_columns if col not in content.columns]
assert not missing_consumer, (f"Missing consumer columns: {missing_consumer}")
assert not missing_content, (f"Missing content columns: {missing_content}")
print("Schema validation passed.")

Schema validation passed.


## 1. Normalize IDs

In [104]:
consumer["consumer_id"] = (consumer["consumer_id"].astype(str).str.strip())
consumer["item_id"] = (consumer["item_id"].astype(str).str.strip())
content["item_id"] = (content["item_id"].astype(str).str.strip())
content["producer_id"] = (content["producer_id"].astype(str).str.strip())
consumer["interaction_type"] = (consumer["interaction_type"].astype(str).str.strip().str.lower())
content["interaction_type"] = (content["interaction_type"].astype(str).str.strip().str.lower())
content["language"] = (content["language"].astype(str).str.strip().str.lower())

## 2. Build Latest Content State

In [105]:
content["event_datetime"] = pd.to_datetime(content["event_timestamp"], unit="s", utc=True)

In [106]:
print("Content columns:")
print(content.columns.tolist())
print("Does event_datetime exist?")
print("event_datetime" in content.columns)

Content columns:
['event_timestamp', 'interaction_type', 'item_id', 'producer_id', 'producer_session_id', 'producer_device_info', 'producer_location', 'producer_country', 'item_type', 'item_url', 'title', 'text_description', 'language', 'event_datetime']
Does event_datetime exist?
True


In [107]:
content["event_timestamp"] = pd.to_numeric( content["event_timestamp"], errors="coerce")
content["event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True,errors="coerce")
print("Content event period:", content["event_datetime"].min(), "→", content["event_datetime"].max())
print("Missing event_datetime:", content["event_datetime"].isna().sum())

Content event period: 2016-03-28 19:19:39+00:00 → 2017-02-28 18:51:11+00:00
Missing event_datetime: 0


In [108]:
content_sorted = content.sort_values(["item_id","event_datetime"]).copy()
latest_content_state = (content_sorted.groupby("item_id",as_index=False).tail(1).copy())
latest_content_state["is_available"] = (latest_content_state["interaction_type"].eq("content_present"))
print("Unique articles in content:", latest_content_state["item_id"].nunique())
print("Currently available:", latest_content_state["is_available"].sum())
print("Currently pulled out:",(~latest_content_state[ "is_available"]).sum())

Unique articles in content: 3057
Currently available: 2983
Currently pulled out: 74


In [109]:
required_columns = ["item_id", "event_timestamp", "interaction_type",]
missing_columns = [column for column in required_columns if column not in content.columns]
if missing_columns:
    raise ValueError( f"Missing required content columns: {missing_columns}")
content["item_id"] = (content["item_id"].astype(str).str.strip())
content["interaction_type"] = (content["interaction_type"].astype(str).str.strip().str.lower())
content["event_timestamp"] = pd.to_numeric(content["event_timestamp"],errors="coerce")
content["event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True,errors="coerce")
missing_datetime = (content["event_datetime"].isna().sum())
if missing_datetime > 0:
    print(f"WARNING: {missing_datetime} "f"content rows have invalid timestamps.")
    content = content[content["event_datetime"].notna()].copy()
content_sorted = (content.sort_values(["item_id","event_datetime"]).copy())
latest_content_state = (content_sorted.groupby("item_id",as_index=False).tail(1).copy())
latest_content_state["is_available"] = (latest_content_state["interaction_type"].eq("content_present"))
available_items = set(latest_content_state.loc[latest_content_state["is_available"],"item_id"])
pulled_items = set(latest_content_state.loc[~latest_content_state["is_available"],"item_id"])
print("CONTENT AVAILABILITY SUMMARY")
print("Content lifecycle rows:",len(content))
print("Unique articles:",latest_content_state["item_id"].nunique())
print("Currently available:",len(available_items))
print("Currently pulled out:",len(pulled_items))
print("Available percentage:",round(len(available_items)/latest_content_state["item_id"].nunique() * 100, 2),"%")

CONTENT AVAILABILITY SUMMARY
Content lifecycle rows: 3122
Unique articles: 3057
Currently available: 2983
Currently pulled out: 74
Available percentage: 97.58 %


## 3. Available & Pulled Out Article

In [110]:
available_items = set(latest_content_state.loc[latest_content_state["is_available"],"item_id"])
pulled_items = set(latest_content_state.loc[~latest_content_state["is_available"],"item_id"])
print("Available items:", len(available_items))
print("Pulled-out items:", len(pulled_items))

Available items: 2983
Pulled-out items: 74


## 4. Bulid Article Catalogs

In [111]:
catalog_columns = ["item_id","producer_id","item_type","item_url","title","text_description","language","producer_country","producer_location","event_datetime","is_available",]
article_catalog = latest_content_state[[col for col in catalog_columns if col in latest_content_state.columns]].copy()
article_catalog["title"] = (article_catalog["title"].fillna("").astype(str).str.strip())
article_catalog["text_description"] = (article_catalog["text_description"].fillna("").astype(str).str.strip())
article_catalog = article_catalog.drop_duplicates(subset=["item_id"])
article_catalog = article_catalog.reset_index(drop=True)
print("Article catalog:", article_catalog.shape)

Article catalog: (3057, 11)


## 5. English Available Article Catalogs

In [112]:
english_available_catalog = article_catalog[article_catalog["is_available"].eq(True) & article_catalog["language"].eq("en")].copy()
english_available_catalog = (english_available_catalog.drop_duplicates("item_id").reset_index(drop=True))
print("English available articles:", len(english_available_catalog))

English available articles: 2166


## 6. Interaction Weights

In [113]:
INTERACTION_WEIGHTS = {"content_watched": 1.0,"content_liked": 2.0,"content_saved": 3.0,"content_followed": 4.0,"content_commented_on": 5.0,}
consumer["interaction_weight"] = (consumer["interaction_type"].map(INTERACTION_WEIGHTS).fillna(0.0))
consumer[["interaction_type","interaction_weight"]].drop_duplicates().sort_values("interaction_weight")

,interaction_type,interaction_weight
0,content_watched,1.0
33,content_liked,2.0
25,content_saved,3.0
3,content_followed,4.0
61,content_commented_on,5.0


## 7. Train / Test Split

In [114]:
required_columns = ["consumer_id","event_timestamp","item_id","interaction_type"]
missing_columns = [col for col in required_columns if col not in consumer.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in consumer: {missing_columns}")
consumer["consumer_id"] = (consumer["consumer_id"].astype(str).str.strip())
consumer["item_id"] = (consumer["item_id"].astype(str).str.strip())
consumer["interaction_type"] = (consumer["interaction_type"].astype(str).str.strip().str.lower())
consumer["event_timestamp"] = pd.to_numeric(consumer["event_timestamp"],errors="coerce")
consumer["event_datetime"] = pd.to_datetime(consumer["event_timestamp"],unit="s",utc=True,errors="coerce")
invalid_timestamps = consumer["event_datetime"].isna().sum()
if invalid_timestamps > 0:
    print(f"Removing {invalid_timestamps:,} rows "f"with invalid timestamps.")
    consumer = consumer[consumer["event_datetime"].notna()].copy()
print("Consumer shape:", consumer.shape)
print("Consumer columns:")
print(consumer.columns.tolist())
print("Timestamp range:")
print("Min:", consumer["event_datetime"].min())
print("Max:", consumer["event_datetime"].max())
print("Timestamp dtype:")
print(consumer["event_datetime"].dtype)
print("Missing event_datetime:")
print(consumer["event_datetime"].isna().sum())

Consumer shape: (72312, 10)
Consumer columns:
['event_timestamp', 'interaction_type', 'item_id', 'consumer_id', 'consumer_session_id', 'consumer_device_info', 'consumer_location', 'country', 'interaction_weight', 'event_datetime']
Timestamp range:
Min: 2016-03-14 13:54:36+00:00
Max: 2017-02-28 19:21:51+00:00
Timestamp dtype:
datetime64[ns, UTC]
Missing event_datetime:
0


In [115]:
consumer = consumer.sort_values(["consumer_id","event_datetime"]).reset_index(drop=True)
user_counts = (consumer.groupby("consumer_id").size())
eligible_users = user_counts[user_counts >= MIN_INTERACTIONS_FOR_PERSONALIZATION].index
eligible_consumer = consumer[consumer["consumer_id"].isin(eligible_users)].copy()
test_indices = (eligible_consumer.groupby("consumer_id").tail(1).index)
test_interactions = (eligible_consumer.loc[test_indices].copy())
train_consumer = (eligible_consumer.drop(index=test_indices).copy())
train_consumer = train_consumer.sort_values("event_datetime").reset_index(drop=True)
test_interactions = test_interactions.sort_values("event_datetime").reset_index(drop=True)
print("Eligible users:", len(eligible_users))
print("Train interactions:", len(train_consumer))
print("Test interactions:", len(test_interactions))
print("Train date range:", train_consumer["event_datetime"].min(), "->", train_consumer["event_datetime"].max())
print("Test date range:",test_interactions["event_datetime"].min(), "->", test_interactions["event_datetime"].max())

Eligible users: 1715
Train interactions: 70417
Test interactions: 1715
Train date range: 2016-03-14 13:54:36+00:00 -> 2017-02-28 18:53:25+00:00
Test date range: 2016-03-30 18:37:49+00:00 -> 2017-02-28 19:21:51+00:00


In [116]:
test_truth = (test_interactions.groupby("consumer_id")["item_id"].apply(set).to_dict())
print("Users in test_truth:",len(test_truth))
print("Test items:",sum(len(items) for items in test_truth.values()))

Users in test_truth: 1715
Test items: 1715


## 8. Bulid User Seen Item Lookup

In [117]:
user_seen_items = (train_consumer.groupby("consumer_id")["item_id"].agg(set).to_dict())
print("Users with training history:",len(user_seen_items))

Users with training history: 1715


## 9. Global Popularity

In [118]:
popularity_stats = (train_consumer.groupby("item_id").agg(interaction_count=("item_id","size"),unique_users=("consumer_id","nunique"),weighted_interactions=("interaction_weight","sum"),last_interaction=("event_datetime","max")).reset_index())
popularity_stats = popularity_stats.merge(article_catalog[["item_id","title","language","is_available","producer_id"]],on="item_id",how="left")
popularity_stats = popularity_stats[popularity_stats["is_available"].eq(True)]
popularity_stats = popularity_stats.sort_values(["weighted_interactions","unique_users","interaction_count"],ascending=False).reset_index(drop=True)
print("Popularity catalog:",popularity_stats.shape)

Popularity catalog: (2911, 9)


## 10. Popularity Score

In [119]:
def min_max_scale(series):
    series = series.astype(float)
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or max_value == min_value:
        return pd.Series(np.ones(len(series)), index=series.index)
    return ((series - min_value) / (max_value - min_value))
popularity_stats["popularity_score"] = (min_max_scale(np.log1p(popularity_stats["weighted_interactions"])))

## 11. Popularity Candidate Generator

In [120]:
def generate_popularity_candidates(user_id,retrieval_k=POPULARITY_K,history_df=train_consumer):
    seen_items = get_seen_items(user_id,history_df)
    candidates = popularity_stats[~popularity_stats["item_id"].isin(seen_items)].head(retrieval_k).copy()
    if candidates.empty:
        return pd.DataFrame(columns=["consumer_id","item_id","popularity_score"])
    candidates["consumer_id"] = user_id
    return candidates[["consumer_id","item_id","popularity_score"]].reset_index(drop=True)

## 12. Trending Candidates

In [121]:
reference_time = train_consumer["event_datetime"].max()
recent_cutoff = (reference_time - pd.Timedelta(days=RECENT_DAYS))
recent_interactions = train_consumer[train_consumer["event_datetime"] >= recent_cutoff].copy()
trending_stats = (recent_interactions.groupby("item_id").agg(recent_interactions=("item_id","size"),recent_unique_users=("consumer_id","nunique"),recent_weighted_interactions=("interaction_weight","sum"),latest_interaction=("event_datetime","max")).reset_index())
trending_stats = trending_stats.merge(article_catalog[["item_id","title","language","is_available"]],on="item_id",how="left")
trending_stats = trending_stats[trending_stats["is_available"].eq(True)].copy()

## 13. Recency Adjusted Treading Score

In [122]:
def recency_weight(event_time,reference_time,half_life_days=RECENCY_HALF_LIFE_DAYS):
    age_days = (reference_time - event_time).total_seconds() / 86400.0
    age_days = max(age_days, 0.0)
    return 0.5 ** (age_days / half_life_days)
recent_interactions["recency_weight"] = (recent_interactions["event_datetime"].apply(lambda x: recency_weight(x,reference_time)))
recent_interactions["recency_weighted_signal"] = (recent_interactions["interaction_weight"] * recent_interactions["recency_weight"])
trending_stats = (recent_interactions.groupby("item_id").agg(recent_interactions=("item_id","size"),recent_unique_users=("consumer_id","nunique"),trending_score=("recency_weighted_signal","sum"),latest_interaction=("event_datetime","max")).reset_index())
trending_stats = trending_stats.merge(article_catalog[["item_id","title","language","is_available"]],on="item_id",how="left")
trending_stats = trending_stats[trending_stats["is_available"].eq(True)].copy()
trending_stats["trending_score"] = min_max_scale(np.log1p(trending_stats["trending_score"]))
trending_stats = trending_stats.sort_values("trending_score",ascending=False).reset_index(drop=True)

## 14. Trending Candidate Generator

In [123]:
def generate_trending_candidates(user_id,retrieval_k=TRENDING_K):
    seen_items = get_seen_items(user_id)
    candidates = trending_stats[~trending_stats["item_id"].isin(seen_items)].head(retrieval_k).copy()
    if candidates.empty:
        return pd.DataFrame(columns=["consumer_id","item_id","trending_score"])
    candidates["consumer_id"] = user_id
    return candidates[["consumer_id","item_id","trending_score"]].reset_index(drop=True)

## 15. Country Popularity

In [124]:
user_country = (train_consumer.sort_values("event_datetime").groupby("consumer_id")["country"].agg(lambda x: (x.dropna().iloc[-1] if not x.dropna().empty else None)).to_dict())
country_popularity = (train_consumer.dropna(subset=["country"]).groupby(["country", "item_id"]).agg(country_interactions=("item_id","size"),country_unique_users=("consumer_id","nunique"),country_weighted_interactions=("interaction_weight","sum")).reset_index())
country_popularity = country_popularity.merge(article_catalog[["item_id","title","language","is_available"]],on="item_id",how="left")
country_popularity = country_popularity[country_popularity["is_available"].eq(True)]
country_popularity["country_score"] = (np.log1p(country_popularity["country_weighted_interactions"]))
country_popularity = (country_popularity.sort_values(["country","country_score"], ascending=False))

## 16. Country Candidate Generator

In [125]:
def generate_country_candidates(user_id,retrieval_k=COUNTRY_K):
    country = user_country.get(str(user_id))
    seen_items = get_seen_items(user_id)
    if country is None:
        return pd.DataFrame(columns=["consumer_id","item_id","country_score"])
    candidates = country_popularity[country_popularity["country"].eq(country) & ~country_popularity["item_id"].isin(seen_items)].head(retrieval_k).copy()
    if candidates.empty:
        return pd.DataFrame(columns=["consumer_id","item_id","country_score"])
    candidates["country_score"] = min_max_scale(candidates["country_score"])
    candidates["consumer_id"] = user_id
    return candidates[["consumer_id","item_id","country_score"]].reset_index(drop=True)

## 17. TF & IDF Catalogs

In [126]:
tfidf_catalog = (english_available_catalog.copy())
tfidf_catalog["article_text"] = ("Title: " + tfidf_catalog["title"] + " Description: " + tfidf_catalog["text_description"])
tfidf_vectorizer = TfidfVectorizer(max_features=10000,stop_words="english",ngram_range=(1, 2),min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(tfidf_catalog["article_text"])
tfidf_matrix = normalize(tfidf_matrix,norm="l2",axis=1)
tfidf_item_to_index = {item_id: index for index, item_id in enumerate(tfidf_catalog["item_id"])}
tfidf_index_to_item = {index: item_id for item_id, index in tfidf_item_to_index.items()}
print("TF-IDF matrix:", tfidf_matrix.shape)

TF-IDF matrix: (2166, 10000)


## 18. Bulid User TF & IDF Profile

In [127]:
def build_user_tfidf_profile(user_id,reference_time,history_df=train_consumer):
    history = history_df[(history_df["consumer_id"] == str(user_id)) & (history_df["event_datetime"] < reference_time)].copy()
    history = history[history["item_id"].isin(tfidf_item_to_index)]
    if history.empty:
        return None
    vectors = []
    weights = []
    for _, row in history.iterrows():
        item_id = row["item_id"]
        index = tfidf_item_to_index.get(item_id)
        if index is None:
            continue
        vector = tfidf_matrix[index]
        recency = recency_weight(row["event_datetime"],reference_time)
        weight = (row["interaction_weight"] * recency)
        if weight <= 0:
            continue
        vectors.append(vector)
        weights.append(weight)
    if not vectors:
        return None
    profile = vectors[0] * weights[0]
    for vector, weight in zip(vectors[1:],weights[1:]):
        profile += vector * weight
    profile = normalize(profile,norm="l2")
    return profile

## 19. TF & IDF Candidate Generator

In [128]:
def generate_tfidf_candidates(user_id,retrieval_k=TFIDF_RETRIEVAL_K,reference_time=None):
    if reference_time is None:
        reference_time = train_consumer["event_datetime"].max()
    user_profile = build_user_tfidf_profile(user_id=user_id,reference_time=reference_time,history_df=train_consumer)
    if user_profile is None:
        return pd.DataFrame(columns=["consumer_id","item_id","tfidf_score"])
    scores = (user_profile @ tfidf_matrix.T).toarray().ravel()
    ranking = np.argsort(-scores)
    seen_items = get_seen_items(user_id)
    rows = []
    for index in ranking:
        item_id = tfidf_index_to_item[int(index)]
        if item_id in seen_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"tfidf_score": float(scores[index])})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 20. Item CF

In [129]:
cf_interactions = (train_consumer.groupby(["consumer_id","item_id"], as_index=False).agg(interaction_strength=("interaction_weight","sum")))
cf_users = (cf_interactions["consumer_id"].drop_duplicates().tolist())
cf_items = (cf_interactions["item_id"].drop_duplicates().tolist())
cf_user_to_index = {user_id: index for index, user_id in enumerate(cf_users)}
cf_item_to_index = {item_id: index for index, item_id in enumerate(cf_items)}
cf_index_to_item = {index: item_id for item_id, index in cf_item_to_index.items()}
rows = [cf_user_to_index[user_id] for user_id in cf_interactions["consumer_id"]]
cols = [cf_item_to_index[item_id] for item_id in cf_interactions["item_id"]]
values = (cf_interactions["interaction_strength"].astype(np.float32))
user_item_matrix = csr_matrix((values,(rows, cols)),shape=(len(cf_users),len(cf_items)))
print("User-item matrix:",user_item_matrix.shape)

User-item matrix: (1715, 2980)


## 21. Item Item Cosine Similarity

In [130]:
from sklearn.metrics.pairwise import cosine_similarity

In [131]:
item_user_matrix = user_item_matrix.T.tocsr()
item_similarity = cosine_similarity(item_user_matrix)
print("Item similarity shape:", item_similarity.shape)

Item similarity shape: (2980, 2980)


## 22. Item CF Candidate Generator

In [132]:
def generate_item_cf_candidates(user_id,retrieval_k=ITEM_CF_RETRIEVAL_K):
    user_id = str(user_id)
    if user_id not in cf_user_to_index:
        return pd.DataFrame(columns=["consumer_id","item_id","cf_score"])
    user_index = cf_user_to_index[user_id]
    user_row = (user_item_matrix[user_index].toarray().ravel())
    interacted_indices = np.flatnonzero(user_row > 0)
    if len(interacted_indices) == 0:
        return pd.DataFrame(columns=["consumer_id","item_id","cf_score"])
    scores = np.zeros(len(cf_items),dtype=np.float32)
    for item_index in interacted_indices:
        interaction_strength = (user_row[item_index])
        scores += (item_similarity[item_index] * interaction_strength)
    seen_items = get_seen_items(user_id)
    ranking = np.argsort(-scores)
    rows = []
    for index in ranking:
        if scores[index] <= 0:
            continue
        item_id = cf_index_to_item[int(index)]
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"cf_score": float(scores[index])})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 23. ALS Matrix

In [133]:
als_user_to_index = cf_user_to_index.copy()
als_item_to_index = cf_item_to_index.copy()
als_index_to_item = cf_index_to_item.copy()
als_matrix = user_item_matrix.astype( np.float32)
print("ALS matrix:", als_matrix.shape)

ALS matrix: (1715, 2980)


## 24. Train ALS

In [134]:
als_model = AlternatingLeastSquares(factors=64,regularization=0.05,iterations=20,random_state=RANDOM_STATE)
als_model.fit(als_matrix)
print("ALS training complete.")

100%|██████████| 20/20 [00:00<00:00, 73.49it/s]

ALS training complete.


## 25. ALS Candidate Generator

In [135]:
def generate_als_candidates(user_id,retrieval_k=ALS_RETRIEVAL_K):
    user_id = str(user_id)
    if user_id not in als_user_to_index:
        return pd.DataFrame(columns=["consumer_id","item_id","als_score"])
    user_index = als_user_to_index[user_id]
    try:
        item_indices, scores = (als_model.recommend(userid=user_index,user_items=als_matrix[user_index],N=retrieval_k * 3,filter_already_liked_items=True))
    except Exception as exc:
        print(f"ALS recommendation failed for "f"user={user_id}: {exc}")
        return pd.DataFrame(columns=["consumer_id","item_id","als_score"])
    rows = []
    for index, score in zip(item_indices,scores):
        item_id = als_index_to_item[int(index)]
        if item_id not in available_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"als_score": float(score)})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 26. Load Semantic Embeddings

In [136]:
ARTICLE_EMBEDDINGS_PATH = (FEATURE_DIR / "article_embeddings.npy")
ARTICLE_EMBEDDING_INDEX_PATH = (FEATURE_DIR / "article_embedding_index.parquet")
FAISS_INDEX_PATH = (INDEX_DIR / "faiss" / "article_cosine.index")
if (ARTICLE_EMBEDDINGS_PATH.exists() and ARTICLE_EMBEDDING_INDEX_PATH.exists() and FAISS_INDEX_PATH.exists()):
    article_embeddings = np.load(ARTICLE_EMBEDDINGS_PATH)
    embedding_metadata = pd.read_parquet(ARTICLE_EMBEDDING_INDEX_PATH)
    faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
    print("Loaded semantic embeddings:", article_embeddings.shape)
    print("Loaded FAISS index:",faiss_index.ntotal)
else:
    print("Semantic artifacts not found.")

Loaded semantic embeddings: (2166, 384)
Loaded FAISS index: 2166


## 27. Semantic Mappings

In [137]:
item_to_embedding_index = {item_id: int(index) for item_id, index in zip(embedding_metadata["item_id"], embedding_metadata["embedding_index"])}
index_to_item = {int(index): item_id for item_id, index in zip(embedding_metadata["item_id"],embedding_metadata["embedding_index"])}
semantic_available_items = set(embedding_metadata["item_id"])
print("Semantic items:", len(semantic_available_items))

Semantic items: 2166


## 28. Recency Weighted Semantic User Embeddings

In [138]:
def build_user_embedding_at_time(user_id, cutoff_time,history_df=train_consumer):
    user_id = str(user_id)
    history = history_df[(history_df["consumer_id"] == user_id) & (history_df["event_datetime"] < cutoff_time)].copy()
    history = history[history["item_id"].isin(item_to_embedding_index)]
    if history.empty:
        return None
    weighted_vectors = []
    total_weight = 0.0
    for _, row in history.iterrows():
        item_id = row["item_id"]
        embedding_index = (item_to_embedding_index.get(item_id))
        if embedding_index is None:
            continue
        interaction_weight = (row["interaction_weight"])
        recency = recency_weight(row["event_datetime"],cutoff_time)
        weight = (interaction_weight  * recency)
        if weight <= 0:
            continue
        vector = article_embeddings[embedding_index].astype(np.float32)
        weighted_vectors.append(vector * weight)
        total_weight += weight
    if not weighted_vectors:
        return None
    user_vector = np.sum(weighted_vectors,axis=0)
    if total_weight <= 0:
        return None
    user_vector /= total_weight
    norm = np.linalg.norm(user_vector)
    if norm == 0:
        return None
    user_vector /= norm
    return user_vector.astype(np.float32)

## 29. Semantic Candidate Generator

In [139]:
def generate_semantic_candidates(user_id,retrieval_k=SEMANTIC_RETRIEVAL_K,cutoff_time=None,history_df=train_consumer):
    if cutoff_time is None:
        cutoff_time = history_df["event_datetime"].max()
    user_vector = build_user_embedding_at_time(user_id=user_id,cutoff_time=cutoff_time,history_df=history_df)
    if user_vector is None:
        return pd.DataFrame(columns=["consumer_id","item_id","semantic_score"])
    search_k = min(retrieval_k * 5,faiss_index.ntotal)
    scores, indices = (faiss_index.search(user_vector.reshape(1, -1).astype(np.float32),search_k))
    seen_items = get_seen_items(user_id)
    rows = []
    for score, index in zip(scores[0],indices[0]):
        if index < 0:
            continue
        item_id = index_to_item[int(index)]
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        if item_id not in semantic_available_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"semantic_score": float(score)})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 30. Validate Each Candidate Generator

In [140]:
def get_seen_items(user_id,history_df=train_consumer):
    if history_df is None or history_df.empty:
        return set()
    user_history = history_df[history_df["consumer_id"] == user_id]
    return set(user_history["item_id"].dropna().astype(str).str.strip())

In [141]:
sample_user = next(iter(test_truth.keys()))
seen_items = get_seen_items(user_id=sample_user,history_df=train_consumer)
print("Sample user:", sample_user)
print("Seen articles:", len(seen_items))
print("First 10 seen items:", list(seen_items)[:10])

Sample user: -1007001694607905623
Seen articles: 5
First 10 seen items: ['-5065077552540450930', '7270966256391553686', '-793729620925729327', '-6623581327558800021', '1469580151036142903']


In [142]:
consumer["consumer_id"] = (consumer["consumer_id"].astype(str).str.strip())
consumer["item_id"] = (consumer["item_id"].astype(str).str.strip())
train_consumer["consumer_id"] = (train_consumer["consumer_id"].astype(str).str.strip())
train_consumer["item_id"] = (train_consumer["item_id"].astype(str).str.strip())
test_interactions["consumer_id"] = (test_interactions["consumer_id"].astype(str).str.strip())
test_interactions["item_id"] = (test_interactions["item_id"].astype(str).str.strip())
content["item_id"] = (content["item_id"].astype(str).str.strip())
print("ID normalization complete.")

ID normalization complete.


In [143]:
sample_user = next(iter(test_truth.keys()))
print("Sample user:", sample_user)
pop_test = generate_popularity_candidates(sample_user, retrieval_k=10)
trend_test = generate_trending_candidates(sample_user,retrieval_k=10)
country_test = generate_country_candidates(sample_user,retrieval_k=10)
tfidf_test = generate_tfidf_candidates(sample_user,retrieval_k=10)
cf_test = generate_item_cf_candidates(sample_user,retrieval_k=10)
als_test = generate_als_candidates(sample_user,retrieval_k=10)
semantic_test = generate_semantic_candidates(sample_user,retrieval_k=10)
print("Popularity:", len(pop_test))
print("Trending:", len(trend_test))
print("Country:", len(country_test))
print("TF-IDF:", len(tfidf_test))
print("Item-CF:", len(cf_test))
print("ALS:", len(als_test))
print("Semantic:", len(semantic_test))

Sample user: -1007001694607905623
Popularity: 10
Trending: 10
Country: 10
TF-IDF: 10
Item-CF: 10
ALS: 10
Semantic: 0


## 31. Candidate Source Registry

In [144]:
CANDIDATE_GENERATORS = {"popularity": generate_popularity_candidates,"trending": generate_trending_candidates,"country": generate_country_candidates,"tfidf": generate_tfidf_candidates,"item_cf": generate_item_cf_candidates,"als": generate_als_candidates,"semantic": generate_semantic_candidates,}

## 32. Generate Candidates For One User

In [145]:
def generate_all_candidates_for_user(user_id,retrieval_k=CANDIDATE_K):
    candidate_frames = []
    generators = {"popularity": generate_popularity_candidates,"trending": generate_trending_candidates,"country": generate_country_candidates,"tfidf": generate_tfidf_candidates,"item_cf": generate_item_cf_candidates,"als": generate_als_candidates,"semantic": generate_semantic_candidates,}
    for source_name, generator in generators.items():
        try:
            candidates = generator(user_id=user_id, retrieval_k=retrieval_k)
        except Exception as exc:
            print(f"[WARNING] " f"{source_name} failed " f"for user {user_id}: {exc}")
            continue
        if candidates.empty:
            continue
        candidates = candidates.copy()
        candidates["source"] = (source_name)
        candidate_frames.append(candidates)
    if not candidate_frames:
        return pd.DataFrame(columns=["consumer_id","item_id","source"])
    return pd.concat(candidate_frames,ignore_index=True)

## 33. Generate Candidates For All Test Users

In [146]:
all_candidate_frames = []
test_users = list(test_truth.keys())
print("Generating candidates for", len(test_users), "users...")
for counter, user_id in enumerate(test_users, start=1):
    candidates = (generate_all_candidates_for_user(user_id=user_id, retrieval_k=CANDIDATE_K))
    if not candidates.empty:
        all_candidate_frames.append(candidates)
    if counter % 100 == 0:
        print(f"Processed {counter}/" f"{len(test_users)} users")
if all_candidate_frames:
    raw_candidates = pd.concat(all_candidate_frames, ignore_index=True)
else:
    raw_candidates = pd.DataFrame(columns=["consumer_id","item_id","source"])
print("Raw candidate rows:", len(raw_candidates))

Generating candidates for 1715 users...
Processed 100/1715 users
Processed 200/1715 users
Processed 300/1715 users
Processed 400/1715 users
Processed 500/1715 users
Processed 600/1715 users
Processed 700/1715 users
Processed 800/1715 users
Processed 900/1715 users
Processed 1000/1715 users
Processed 1100/1715 users
Processed 1200/1715 users
Processed 1300/1715 users
Processed 1400/1715 users
Processed 1500/1715 users
Processed 1600/1715 users
Processed 1700/1715 users
Raw candidate rows: 924719


## 34. Inspect Source Coverages

In [147]:
source_coverage = (raw_candidates.groupby("source").agg(candidate_rows=("item_id","size"),users=("consumer_id","nunique"),unique_items=("item_id","nunique")).sort_values("candidate_rows",ascending=False))
display(source_coverage)

,candidate_rows,users,unique_items
source,,,
als,171500,1715,2277
popularity,171500,1715,191
item_cf,171182,1714,2884
country,169212,1700,511
tfidf,148100,1481,2159
trending,93225,1715,55


## 35. Convert Source Rows Into Wide Candidate Features

In [148]:
score_columns = ["popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score",]
available_score_columns = [col for col in score_columns if col in raw_candidates.columns]
candidate_scores = (raw_candidates.groupby(["consumer_id","item_id"],as_index=False)[available_score_columns].max())
print("Unique user-item candidates:", len(candidate_scores))

Unique user-item candidates: 666597


In [149]:
source_presence = (raw_candidates[["consumer_id","item_id","source"]].drop_duplicates().assign(present=1).pivot_table(index=["consumer_id","item_id"],columns="source",values="present",fill_value=0).reset_index())
source_presence.columns.name = None
source_presence = source_presence.rename(columns={"popularity": "from_popularity","trending": "from_trending","country": "from_country","tfidf": "from_tfidf","item_cf": "from_item_cf","als":
 "from_als", "semantic": "from_semantic"})
candidate_scores = candidate_scores.merge(source_presence,on=["consumer_id","item_id"],how="left")

## 36. Fill Missing Source Scores

In [150]:
for column in score_columns:
    if column not in candidate_scores.columns:
        candidate_scores[column] = 0.0
    candidate_scores[column] = (candidate_scores[column].fillna(0.0).astype(float))
source_flags = ["from_popularity","from_trending","from_country","from_tfidf","from_item_cf","from_als","from_semantic",]
for column in source_flags:
    if column not in candidate_scores.columns:
        candidate_scores[column] = 0
    candidate_scores[column] = (candidate_scores[column].fillna(0).astype(int))

## 37. Number Of Candidates Sources

In [151]:
candidate_scores["num_sources"] = (candidate_scores[source_flags].sum(axis=1))
candidate_scores["multi_source_candidate"] = (candidate_scores["num_sources"] >= 2).astype(int)

## 38. Filtering : Seen Articles & Unavailable Articles

In [152]:
before_seen_filter = len(candidate_scores)
candidate_scores = candidate_scores[candidate_scores.apply(lambda row: (row["item_id"] not in get_seen_items(row["consumer_id"])),axis=1)].copy()
after_seen_filter = len(candidate_scores)
print("Removed seen candidates:", before_seen_filter - after_seen_filter)

Removed seen candidates: 0


In [153]:
before_availability_filter = len(candidate_scores)
candidate_scores = candidate_scores[candidate_scores["item_id"].isin(available_items)].copy()
after_availability_filter = len(candidate_scores)
print("Removed unavailable candidates:", before_availability_filter - after_availability_filter)

Removed unavailable candidates: 0
